# See it for yourself

**This needs nothing from you.** No data, no accounts, no setup. It makes its own
test data and runs in about five minutes.

The question it answers is not "does this system make money" — that needs real
price history, which is Step 2 of your life, not this notebook. The question here
is **"is this machinery honest?"** Specifically:

1. Does the code actually work, or does it just claim to?
2. When there is no edge in the data, does the backtest correctly find nothing —
   or does it manufacture a win rate?
3. Does the validator refuse bad results *and* accept good ones? A validator that
   always says no is as useless as one that always says yes.
4. Did making the backtest ~6,000× faster change any of its answers?

Run **Runtime → Run all** and read down the page.

---

*Why this notebook exists: the validator in this repo was, at one point, passing
pure random noise and declaring it a survivor — which was enough to unlock live
trading. That was found by doing exactly what this notebook does. It seemed worth
letting you run the same check yourself rather than taking my word for it.*

## Step 1 — Get the code and run its tests

417 tests, no network needed. If these fail, nothing below is worth reading.

In [ ]:
import os, shutil, subprocess

REPO = "https://github.com/dboy140/Dboytrades.git"
BRANCH = "claude/ict-nbbtrader-trading-system-43hipg"

if os.path.isdir("/content/Dboytrades"):
    shutil.rmtree("/content/Dboytrades")
r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO,
                    "/content/Dboytrades"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit(f"Could not download the code:\n{r.stderr}")
os.chdir("/content/Dboytrades")

%pip install -q pydantic pytest
t = subprocess.run(["python", "-m", "pytest", "-q"], capture_output=True, text=True)
print(t.stdout.strip().splitlines()[-1] if t.stdout else t.stderr[-400:])

## Step 2 — Make data with no edge in it

A random walk. Each minute moves by a coin flip, so there is nothing to find —
any strategy tested on this *should* come out flat. If a backtest reports a good
win rate here, the backtest is broken.

This is not a hypothetical concern. Testing on random data is what caught a bug
in this repo where a long's stop was placed **above** its entry, triggering
instantly and booking a profit every time. It reported a **100% win rate** and
looked wonderful.

In [ ]:
import random
from datetime import datetime, timedelta, timezone
from bot.bars import Bar

def random_walk(n, seed=3):
    r = random.Random(seed)
    t = datetime(2024, 1, 1, tzinfo=timezone.utc)
    px, out, k = 1.1000, [], 0
    while k < n:
        if t.weekday() < 5:                      # weekdays only, like a real market
            o = px; px += r.gauss(0, 0.00035); c = px
            h = max(o, c) + abs(r.gauss(0, 0.00018))
            l = min(o, c) - abs(r.gauss(0, 0.00018))
            out.append(Bar(t, o, h, l, c, 100.0)); k += 1
        t += timedelta(minutes=1)
    return out

bars = random_walk(150_000)
print(f"{len(bars):,} one-minute bars, "
      f"{bars[0].ts.date()} to {bars[-1].ts.date()}")
print("There is no pattern in this data. Nothing should be found.")

## Step 3 — Backtest it

Fills are pessimistic on purpose. Entries are limit orders, not market orders,
because that is what the rules describe. When one bar contains both the stop and
the target, the stop is assumed. Excursion is capped at the exit, so a 1R loss
reports as −1R rather than however far price ran afterwards.

**Expect roughly break-even.** Expectancy near 0.00R and profit factor near 1.0
is the correct answer here, and it is what a working engine looks like.

In [ ]:
from bot.backtest import run
from bot.bars import SwingIndexCache
from bot.signals import ote

cache = SwingIndexCache()
res = run(bars, lambda bs, i: ote(bs, i, "EURUSD", index=cache.get(bs)))
s = res.stats()

for k in ("trades", "win_rate", "expectancy_r", "profit_factor",
          "total_r", "max_drawdown_r", "avg_mae_r", "avg_mfe_r"):
    print(f"  {k:<18} {s.get(k)}")

print()
e = s.get("expectancy_r") or 0
print("Expectancy is near zero -- correct." if abs(e) < 0.25 else
      f"Expectancy {e}: on random data this deserves suspicion, not celebration.")

## Step 4 — The important one: can the validator tell the difference?

Anyone can write a checker that rejects everything. The test that matters is
whether it **discriminates** — refuses a result that only looks good, and accepts
one that genuinely is.

Two samples of out-of-sample trades, both positive on average:

- **No edge**: a coin weighted to 42% wins at 1:2 reward-to-risk. Positive
  average purely by the luck of the draw, on 60 trades.
- **Real edge**: 55% at 1:2, sustained across every window, on 200 trades.

A naive check ("is the average positive?") passes both. That is precisely the
mistake this validator used to make.

In [ ]:
import random, statistics
from bot.validate import WalkForwardReport, Fold

def report_from(folds_of_r, in_sample=1.0):
    rep = WalkForwardReport()
    for i, rs in enumerate(folds_of_r):
        rep.folds.append(Fold(i, "a", "b", "c", "d", {}, in_sample,
                              round(statistics.mean(rs), 4), len(rs), rs))
    return rep

def coin(p, n, seed):
    r = random.Random(seed)
    return [2.0 if r.random() < p else -1.0 for _ in range(n)]

no_edge   = report_from([coin(0.42, 15, 100 + i) for i in range(4)])
real_edge = report_from([coin(0.55, 50, 200 + i) for i in range(4)])

for name, rep in (("NO EDGE (42% win rate)", no_edge),
                  ("REAL EDGE (55% win rate)", real_edge)):
    ci = rep.oos_confidence
    print(f"{name}")
    print(f"  weighted expectancy   {rep.combined_oos_expectancy:+.3f}R "
          f"over {rep.total_oos_trades} trades   <-- both positive")
    print(f"  95% interval          [{ci.get('ci95_low')}, {ci.get('ci95_high')}]")
    print(f"  excludes zero         {ci.get('positive_with_95pct_confidence')}")
    print(f"  profitable windows    {rep.folds_positive}/{rep.folds_scored}")
    print(f"  VERDICT               {rep.verdict[:80]}")
    print()

### What just happened

Both samples had a **positive average**. The old check looked only at that, and
would have passed both — and passing the first one is what let a random walk
unlock live trading.

The 95% interval is what separates them. On the no-edge sample it straddles zero,
meaning "a system with no edge produces results like this routinely." On the real
sample it sits entirely above zero.

This is also why a promised "high win rate" should make you suspicious rather than
pleased. A win rate is trivial to manufacture — widen the stop, shrink the target,
and you can hit 90% while losing money steadily. Nothing in this repo optimises
for win rate for that reason.

## Step 5 — Now run the real validator on the random walk

End to end this time: walk-forward across folds, parameters chosen on each
training block and traded untouched on the block that follows.

Whatever it prints, it must **not** unlock live trading.

One thing to expect: on a random walk the strategy barely trades at all. The
entry conditions need a market structure shift inside a session window with a
displacement and a gap, and noise rarely obliges. So the most likely verdict is
`INSUFFICIENT DATA` — which is the honest answer, not a dodge. There is a second
guard for exactly this reason: with only a handful of out-of-sample trades a
bootstrap interval can come out misleadingly narrow, so the trade-count floor
catches what the interval alone would not.

In [ ]:
import csv, pathlib
pathlib.Path("data/bars").mkdir(parents=True, exist_ok=True)
p = "data/bars/random_walk.csv"
with open(p, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["timestamp", "open", "high", "low", "close", "volume"])
    for b in bars:
        w.writerow([b.ts.isoformat(), b.open, b.high, b.low, b.close, b.volume])

import subprocess
out = subprocess.run(["python", "-m", "bot.run_validate", p,
                      "--instrument", "EURUSD", "--setup", "ote",
                      "--out", "logs/rw.json"], capture_output=True, text=True)
print(out.stdout or out.stderr)

In [ ]:
import json, pathlib
from bot.live import validation_allows_live

d = json.loads(pathlib.Path("logs/rw.json").read_text())
ci = d.get("oos_confidence", {})

print("VERDICT:", d["verdict"])
print()
print(f"  out-of-sample trades  {d['total_oos_trades']}")
print(f"  weighted expectancy   {d['combined_oos_expectancy']:+.3f}R")
print(f"  95% interval          [{ci.get('ci95_low')}, {ci.get('ci95_high')}]")
print(f"  profitable windows    {d.get('folds_positive')}/{d.get('folds_scored')}")
print()

ok, why = validation_allows_live("logs/rw.json")
print("Would this unlock live trading?", "YES" if ok else "NO")
print("Reason:", why)
print()
if ok:
    print("YES on a random walk means something is wrong. Please tell me --")
    print("that is the failure mode this whole notebook exists to catch.")
else:
    print("NO is the correct answer. There is no edge in a random walk,")
    print("and the gate is designed so that every guard has to pass, not just one.")

## Step 6 — Did making it faster change any answers?

The backtest used to rescan the whole price history on every single bar. On two
years of 1-minute data that extrapolated to roughly **140 hours** — not slow, but
unrunnable.

It now indexes the structure once. Same work, ~99 microseconds per bar, about
**75 seconds** for the same file.

A speed-up that quietly changes the result would be far worse than a slow
backtest. So: run the same data through both paths and compare the trades.

In [ ]:
from bot.bars import confirmed_swings
import time

sample = random_walk(5000, seed=21)

cache = SwingIndexCache()
t0 = time.time()
fast = run(sample, lambda bs, i: ote(bs, i, "EURUSD", index=cache.get(bs)))
t_fast = time.time() - t0

# The old behaviour: rescan everything, every bar. The window default binds at
# definition time, so the function object itself has to be patched.
original = confirmed_swings.__defaults__
confirmed_swings.__defaults__ = (2, None, None)
try:
    t0 = time.time()
    slow = run(sample, lambda bs, i: ote(bs, i, "EURUSD"))
    t_slow = time.time() - t0
finally:
    confirmed_swings.__defaults__ = original

shape = lambda t: (t.entry_index, t.entry, t.stop, t.target, t.exit_reason)
same = [shape(t) for t in fast.trades] == [shape(t) for t in slow.trades]

print(f"  indexed   {len(fast.trades)} trades in {t_fast:.2f}s")
print(f"  old way   {len(slow.trades)} trades in {t_slow:.2f}s  "
      f"({t_slow / max(t_fast, 1e-9):.0f}x slower)")
print()
print(f"  Identical trades: {same}")
print("  (on 5,000 bars. The gap widens quadratically -- at 750,000 it was 140 hours.)"
      if same else "  MISMATCH -- please tell me, this is a bug.")

---

## What this did and did not show

**Shown:** the code runs and its tests pass; the backtest finds no edge in data
that has none; the validator distinguishes a lucky sample from a real one and
refuses the lucky one; and the fast path returns identical trades to the slow one.

**Not shown — and this is the whole remaining question:** whether the ICT and
NBBTRADER rules make money on *real* price history. Nothing here can tell you
that, because a random walk is not a market.

## What would actually answer it

Two years of 1-minute bars for one instrument. Free, from your own broker:

**MT5** → View → Symbols → pick the symbol → Bars tab → set the date range →
Export. **MT4** → Tools → History Center → symbol → 1 Minute → Export.

Then open `notebooks/validate_colab.ipynb`, upload the file, and run it.

⚠️ **One trap worth knowing about.** MT4/MT5 export in *broker server time*,
usually UTC+2 or UTC+3 — not UTC. Label that as UTC and every session window in
this system is wrong by two or three hours, and the backtest will run anyway and
print confident numbers. Step 4 of the validation notebook detects this and tells
you how to correct it. Do not skip it.

Two years is not me being precious. The trade windows are about an hour a day and
a setup does not appear daily, so a month of data yields a handful of trades —
and as Step 4 above showed, a handful of trades produces a confident wrong answer
rather than a cautious one.